In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load all tables
app_train = pd.read_csv('../data/raw/application_train.csv', encoding='cp1252')
app_test  = pd.read_csv('../data/raw/application_test.csv',  encoding='cp1252')
bureau    = pd.read_csv('../data/raw/bureau.csv',            encoding='cp1252')
bb        = pd.read_csv('../data/raw/bureau_balance.csv',    encoding='cp1252')
prev      = pd.read_csv('../data/raw/previous_application.csv', encoding='cp1252')
inst      = pd.read_csv('../data/raw/installments_payments.csv', encoding='cp1252')
cc        = pd.read_csv('../data/raw/credit_card_balance.csv',   encoding='cp1252')
pos       = pd.read_csv('../data/raw/POS_CASH_balance.csv',      encoding='cp1252')

print("Shapes:")
for name, df in zip(['app_train','bureau','bb','prev','inst','cc','pos'],
                    [app_train, bureau, bb, prev, inst, cc, pos]):
    print(f"  {name}: {df.shape}")

Shapes:
  app_train: (307511, 122)
  bureau: (1716428, 17)
  bb: (27299925, 3)
  prev: (1670214, 37)
  inst: (13605401, 8)
  cc: (3840312, 23)
  pos: (10001358, 8)


In [2]:
# Encode STATUS as numeric severity
status_map = {'C': 0, 'X': 0, '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}
bb['STATUS_NUM'] = bb['STATUS'].map(status_map)

bb_agg = bb.groupby('SK_ID_BUREAU').agg(
    bb_months_count   = ('MONTHS_BALANCE', 'count'),
    bb_max_dpd        = ('STATUS_NUM', 'max'),   # worst ever delinquency
    bb_mean_dpd       = ('STATUS_NUM', 'mean'),  # avg delinquency
    bb_dpd_months     = ('STATUS_NUM', lambda x: (x > 0).sum()),  # months with any DPD
).reset_index()

print(bb_agg.shape)
bb_agg.head()

(817395, 5)


,SK_ID_BUREAU,bb_months_count,bb_max_dpd,bb_mean_dpd,bb_dpd_months
0,5001709,97,0,0.0,0
1,5001710,83,0,0.0,0
2,5001711,4,0,0.0,0
3,5001712,19,0,0.0,0
4,5001713,22,0,0.0,0


In [3]:
# Join bureau with bureau_balance aggregates
bureau_full = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')

bureau_agg = bureau_full.groupby('SK_ID_CURR').agg(
    # Credit counts
    bureau_loan_count         = ('SK_ID_BUREAU', 'count'),
    bureau_active_count       = ('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    bureau_closed_count       = ('CREDIT_ACTIVE', lambda x: (x == 'Closed').sum()),

    # Credit amounts
    bureau_debt_sum           = ('AMT_CREDIT_SUM_DEBT', 'sum'),
    bureau_credit_sum         = ('AMT_CREDIT_SUM', 'sum'),
    bureau_overdue_sum        = ('AMT_CREDIT_SUM_OVERDUE', 'sum'),

    # Delinquency history
    bureau_max_overdue        = ('AMT_CREDIT_MAX_OVERDUE', 'max'),
    bureau_days_credit_mean   = ('DAYS_CREDIT', 'mean'),
    bureau_days_enddate_max   = ('DAYS_CREDIT_ENDDATE', 'max'),

    # From bureau_balance
    bureau_bb_max_dpd         = ('bb_max_dpd', 'max'),
    bureau_bb_mean_dpd        = ('bb_mean_dpd', 'mean'),
    bureau_bb_dpd_months_sum  = ('bb_dpd_months', 'sum'),
).reset_index()

print(bureau_agg.shape)
bureau_agg.head()

(305811, 13)


,SK_ID_CURR,bureau_loan_count,bureau_active_count,bureau_closed_count,bureau_debt_sum,bureau_credit_sum,bureau_overdue_sum,bureau_max_overdue,bureau_days_credit_mean,bureau_days_enddate_max,bureau_bb_max_dpd,bureau_bb_mean_dpd,bureau_bb_dpd_months_sum
0,100001,7,3,4,596686.5,1453365.000,0.0,NaN,-735.000000,1778.0,1.0,0.007519,1.0
1,100002,8,2,6,245781.0,865055.565,0.0,5043.645,-874.000000,780.0,1.0,0.255682,27.0
2,100003,4,1,3,0.0,1017400.500,0.0,0.000,-1400.750000,1216.0,NaN,NaN,0.0
3,100004,2,0,2,0.0,189037.800,0.0,0.000,-867.000000,-382.0,NaN,NaN,0.0
4,100005,3,2,1,568408.5,657126.000,0.0,0.000,-190.666667,1324.0,0.0,0.000000,0.0


In [4]:
prev[['DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION']].describe()

,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION
count,997149.000000,997149.000000,997149.000000,997149.000000,997149.000000
mean,342209.855039,13826.269337,33767.774054,76582.403064,81992.343838
std,88916.115833,72444.869708,106857.034789,149647.415123,153303.516729
min,-2922.000000,-2892.000000,-2801.000000,-2889.000000,-2874.000000
25%,365243.000000,-1628.000000,-1242.000000,-1314.000000,-1270.000000
50%,365243.000000,-831.000000,-361.000000,-537.000000,-499.000000
75%,365243.000000,-411.000000,129.000000,-74.000000,-44.000000
max,365243.000000,365243.000000,365243.000000,365243.000000,365243.000000


In [5]:
# Replace anomalous values
prev['DAYS_FIRST_DRAWING'].replace(365243, np.nan, inplace=True)
prev['DAYS_FIRST_DUE'].replace(365243, np.nan, inplace=True)
prev['DAYS_LAST_DUE_1ST_VERSION'].replace(365243, np.nan, inplace=True)
prev['DAYS_LAST_DUE'].replace(365243, np.nan, inplace=True)
prev['DAYS_TERMINATION'].replace(365243, np.nan, inplace=True)

# Credit utilization ratio for previous loans
prev['PREV_CREDIT_UTIL'] = prev['AMT_APPLICATION'] / (prev['AMT_CREDIT'] + 1)

prev_agg = prev.groupby('SK_ID_CURR').agg(
    prev_app_count             = ('SK_ID_PREV', 'count'),
    prev_approved_count        = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
    prev_refused_count         = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
    prev_amt_credit_mean       = ('AMT_CREDIT', 'mean'),
    prev_amt_annuity_mean      = ('AMT_ANNUITY', 'mean'),
    prev_amt_down_payment_mean = ('AMT_DOWN_PAYMENT', 'mean'),
    prev_credit_util_mean      = ('PREV_CREDIT_UTIL', 'mean'),
    prev_days_decision_min     = ('DAYS_DECISION', 'min'),
).reset_index()

# Approval rate
prev_agg['prev_approval_rate'] = (
    prev_agg['prev_approved_count'] / prev_agg['prev_app_count']
)

print(prev_agg.shape)
prev_agg.head()

(338857, 10)


,SK_ID_CURR,prev_app_count,prev_approved_count,prev_refused_count,prev_amt_credit_mean,prev_amt_annuity_mean,prev_amt_down_payment_mean,prev_credit_util_mean,prev_days_decision_min,prev_approval_rate
0,100001,1,1,0,23787.00,3951.000,2520.0,1.044035,-1740,1.0
1,100002,1,1,0,179055.00,9251.775,0.0,0.999994,-606,1.0
2,100003,3,3,0,484191.00,56553.990,3442.5,0.949323,-2341,1.0
3,100004,1,1,0,20106.00,5357.250,4860.0,1.207639,-815,1.0
4,100005,2,1,0,20076.75,4813.200,4464.0,0.555573,-757,0.5


In [6]:
# Payment behavior: did they pay on time, how much?
inst['DAYS_LATE'] = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
inst['AMT_UNDERPAID'] = inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']

inst_agg = inst.groupby('SK_ID_CURR').agg(
    inst_count                = ('SK_ID_PREV', 'count'),
    inst_days_late_mean       = ('DAYS_LATE', 'mean'),
    inst_days_late_max        = ('DAYS_LATE', 'max'),
    inst_late_payments_count  = ('DAYS_LATE', lambda x: (x > 0).sum()),
    inst_amt_underpaid_mean   = ('AMT_UNDERPAID', 'mean'),
    inst_amt_underpaid_sum    = ('AMT_UNDERPAID', 'sum'),
).reset_index()

# Late payment rate
inst_agg['inst_late_payment_rate'] = (
    inst_agg['inst_late_payments_count'] / inst_agg['inst_count']
)

print(inst_agg.shape)
inst_agg.head()

(339587, 8)


,SK_ID_CURR,inst_count,inst_days_late_mean,inst_days_late_max,inst_late_payments_count,inst_amt_underpaid_mean,inst_amt_underpaid_sum,inst_late_payment_rate
0,100001,7,-7.285714,11.0,1,0.0,0.0,0.142857
1,100002,19,-20.421053,-12.0,0,0.0,0.0,0.000000
2,100003,25,-7.160000,-1.0,0,0.0,0.0,0.000000
3,100004,3,-7.666667,-3.0,0,0.0,0.0,0.000000
4,100005,9,-23.555556,1.0,1,0.0,0.0,0.111111


In [7]:
cc['CC_UTIL'] = cc['AMT_BALANCE'] / (cc['AMT_CREDIT_LIMIT_ACTUAL'] + 1)

cc['AMT_DRAWINGS_TOTAL'] = cc['AMT_DRAWINGS_ATM_CURRENT'] + cc['AMT_DRAWINGS_CURRENT'] + cc['AMT_DRAWINGS_OTHER_CURRENT'] + cc['AMT_DRAWINGS_POS_CURRENT']

cc['AMT_PAYMENT_TOTAL_CURRENT'] = cc['AMT_PAYMENT_CURRENT'] + cc['AMT_PAYMENT_TOTAL_CURRENT']

cc_agg = cc.groupby('SK_ID_CURR').agg(
    cc_count              = ('SK_ID_PREV', 'count'),
    cc_balance_mean       = ('AMT_BALANCE', 'mean'),
    cc_balance_max        = ('AMT_BALANCE', 'max'),
    cc_drawings_mean      = ('AMT_DRAWINGS_TOTAL', 'mean'),
    cc_payment_rate_mean  = ('AMT_PAYMENT_TOTAL_CURRENT', 'mean'),
    cc_util_mean          = ('CC_UTIL', 'mean'),
    cc_util_max           = ('CC_UTIL', 'max'),
    cc_dpd_max            = ('SK_DPD', 'max'),
    cc_dpd_mean           = ('SK_DPD', 'mean'),
).reset_index()

print(cc_agg.shape)
cc_agg.head()

(103558, 10)


,SK_ID_CURR,cc_count,cc_balance_mean,cc_balance_max,cc_drawings_mean,cc_payment_rate_mean,cc_util_mean,cc_util_max,cc_dpd_max,cc_dpd_mean
0,100006,6,0.000000,0.00,NaN,NaN,0.000000,0.000000,0,0.000000
1,100011,74,54482.111149,189000.00,4864.864865,9363.131757,0.302677,1.049994,0,0.000000
2,100013,96,18159.919219,161420.22,12700.000000,13985.518594,0.115300,1.024884,1,0.010417
3,100021,17,0.000000,0.00,NaN,NaN,0.000000,0.000000,0,0.000000
4,100023,8,0.000000,0.00,NaN,NaN,0.000000,0.000000,0,0.000000


In [8]:
pos_agg = pos.groupby('SK_ID_CURR').agg(
    pos_count             = ('SK_ID_PREV', 'count'),
    pos_months_balance    = ('MONTHS_BALANCE', 'mean'),
    pos_cnt_instalment    = ('CNT_INSTALMENT', 'mean'),
    pos_dpd_max           = ('SK_DPD', 'max'),
    pos_dpd_mean          = ('SK_DPD', 'mean'),
    pos_dpd_nonzero       = ('SK_DPD', lambda x: (x > 0).sum()),
    pos_completed_count   = ('NAME_CONTRACT_STATUS',
                             lambda x: (x == 'Completed').sum()),
).reset_index()

print(pos_agg.shape)
pos_agg.head()

(337252, 8)


,SK_ID_CURR,pos_count,pos_months_balance,pos_cnt_instalment,pos_dpd_max,pos_dpd_mean,pos_dpd_nonzero,pos_completed_count
0,100001,9,-72.555556,4.000000,7,0.777778,1,2
1,100002,19,-10.000000,24.000000,0,0.000000,0,0
2,100003,28,-43.785714,10.107143,0,0.000000,0,2
3,100004,4,-25.500000,3.750000,0,0.000000,0,1
4,100005,11,-20.000000,11.700000,0,0.000000,0,1


In [9]:
master = app_train.copy()

for agg_df, name in zip(
    [bureau_agg, prev_agg, inst_agg, cc_agg, pos_agg],
    ['bureau', 'prev', 'inst', 'cc', 'pos']
):
    before = master.shape[1]
    master = master.merge(agg_df, on='SK_ID_CURR', how='left')
    after = master.shape[1]
    print(f"After joining {name}: {master.shape} (+{after - before} features)")

print(f"\nFinal master table shape: {master.shape}")

After joining bureau: (307511, 134) (+12 features)
After joining prev: (307511, 143) (+9 features)
After joining inst: (307511, 150) (+7 features)
After joining cc: (307511, 159) (+9 features)
After joining pos: (307511, 166) (+7 features)

Final master table shape: (307511, 166)


In [11]:
display(master['TARGET'].value_counts())

# Check new missing values from joins (NaN = applicant has no history in that table)
new_missing = master.isnull().mean() * 100
new_missing = new_missing[new_missing > 0].sort_values(ascending=False)
print(f"\nFeatures with missing values: {len(new_missing)}")
print(new_missing.head(20))

# Save
import os
os.makedirs('../data/processed', exist_ok=True)
master.to_parquet('../data/processed/master_table.parquet', engine='fastparquet', index=False)

print(f"Final shape: {master.shape}")

TARGET
0    282686
1     24825
Name: count, dtype: int64


Features with missing values: 111
cc_payment_rate_mean        80.143800
cc_drawings_mean            80.117784
cc_balance_mean             71.739222
cc_util_mean                71.739222
cc_balance_max              71.739222
cc_dpd_max                  71.739222
cc_util_max                 71.739222
cc_count                    71.739222
cc_dpd_mean                 71.739222
bureau_bb_mean_dpd          70.007252
bureau_bb_max_dpd           70.007252
COMMONAREA_MEDI             69.872297
COMMONAREA_AVG              69.872297
COMMONAREA_MODE             69.872297
NONLIVINGAPARTMENTS_AVG     69.432963
NONLIVINGAPARTMENTS_MEDI    69.432963
NONLIVINGAPARTMENTS_MODE    69.432963
FONDKAPREMONT_MODE          68.386172
LIVINGAPARTMENTS_MEDI       68.354953
LIVINGAPARTMENTS_MODE       68.354953
dtype: float64
Final shape: (307511, 166)


## Feature Engineering Summary

- **Source tables joined:** 7 (bureau, bureau_balance, previous_application,
  installments_payments, credit_card_balance, POS_CASH_balance)
- **Final master table:** (307511, 166)
- **Key engineered features:**
  - `bureau_bb_max_dpd` — worst ever delinquency in credit bureau history
  - `inst_late_payment_rate` — share of late installment payments
  - `prev_approval_rate` — ratio of approved to total previous applications
  - `cc_util_max` — peak credit card utilization
  - `pos_dpd_nonzero` — number of months with any DPD in POS loans
- **NaN after join** = applicant has no history in that table — will be treated
  as a separate category in WoE binning (notebook 04)

**Next step:** WoE/IV feature selection → notebook 04 